In [3]:
def brahmi_to_iast(text):
    """Convert Brahmi script to IAST Latin transliteration.

    Transliterate Brahmi to IAST, handling vowels, consonants, conjuncts, diacritics,
    punctuation, and numerals.

    Args:
        text (str): Brahmi script text.

    Returns:
        str: IAST transliterated text.

    Raises:
        ValueError: If input contains invalid Unicode characters.

    Examples:
        >>> brahmi_to_iast("𑀫𑀫 𑀦𑀸𑀫")
        'mama nāma'
        >>> brahmi_to_iast("𑁦𑁧𑁨")
        '012'
    """
    if not text:
        return ""

    # Character mappings for Brahmi to IAST
    vowels = {
        '\U00011005': 'a', '\U00011006': 'ā', '\U00011007': 'i', '\U00011008': 'ī',
        '\U00011009': 'u', '\U0001100A': 'ū', '\U0001100B': 'ṛ', '\U0001100C': 'ṝ',
        '\U0001100D': 'ḷ', '\U0001100E': 'e', '\U0001100F': 'ai', '\U00011010': 'o',
        '\U00011011': 'au'
    }
    consonants = {
        '\U00011013': 'k', '\U00011014': 'kh', '\U00011015': 'g', '\U00011016': 'gh',
        '\U00011017': 'ṅ', '\U00011018': 'c', '\U00011019': 'ch', '\U0001101A': 'j',
        '\U0001101B': 'jh', '\U0001101C': 'ñ', '\U0001101D': 'ṭ', '\U0001101E': 'ṭh',
        '\U0001101F': 'ḍ', '\U00011020': 'ḍh', '\U00011021': 'ṇ', '\U00011022': 't',
        '\U00011023': 'th', '\U00011024': 'd', '\U00011025': 'dh', '\U00011026': 'n',
        '\U00011027': 'p', '\U00011028': 'ph', '\U00011029': 'b', '\U0001102A': 'bh',
        '\U0001102B': 'm', '\U0001102C': 'y', '\U0001102D': 'r', '\U0001102E': 'l',
        '\U0001102F': 'v', '\U00011030': 'ś', '\U00011031': 'ṣ', '\U00011032': 's',
        '\U00011033': 'h'
    }
    vowel_signs = {
        '\U00011038': 'ā', '\U00011039': 'i', '\U0001103A': 'i', '\U0001103B': 'ī',
        '\U0001103C': 'u', '\U0001103D': 'ū', '\U0001103E': 'ṛ', '\U0001103F': 'ṝ',
        '\U00011040': 'ḷ', '\U00011041': 'e', '\U00011042': 'e', '\U00011043': 'o',
        '\U00011044': 'au'
    }
    diacritics = {
        '\U00011000': 'ṃ', '\U00011001': 'ṃ', '\U00011002': 'ḥ', '\U00011046': ''
    }
    punctuation = {
        '\U0001104B': '॥', '\U0001104C': '।', '\U00011047': '.', '\U00011048': '..'
    }
    numerals = {
        '\U00011066': '0', '\U00011067': '1', '\U00011068': '2', '\U00011069': '3',
        '\U0001106A': '4', '\U0001106B': '5', '\U0001106C': '6', '\U0001106D': '7',
        '\U0001106E': '8', '\U0001106F': '9'
    }
    conjuncts = {
        '\U00011027\U00011046\U0001102D': 'pra',
        '\U00011022\U00011046\U0001102C': 'tya',
        '\U00011022\U00011046\U00011027': 'tp',
        '\U00011018\U00011046\U00011019': 'cch'
    }
    orthographic_corrections = {
        '\U00011044': ('au', 'o'),
        '\U0001103F\U00011046': ('ṝ', 'ṛ'),
        '\U0001100B\U00011046': ('ṛ', 'r')
    }

    # Combine mappings for efficient lookup
    all_chars = {**vowels, **consonants, **vowel_signs, **diacritics, **punctuation,
                 **numerals}

    try:
        words = text.split()
        result = []

        for word in words:
            translated_word = ""
            i = 0
            # Check for final virama to suppress inherent 'a'
            has_final_virama = len(word) >= 2 and word[-1] == '\U00011046'
            word_to_process = word[:-1] if has_final_virama else word

            while i < len(word_to_process):
                # Handle predefined conjuncts (e.g., pra, tya)
                if (i + 2 < len(word_to_process) and
                        word_to_process[i:i+3] in conjuncts):
                    translated_word += conjuncts[word_to_process[i:i+3]]
                    i += 3
                    # Add vowel sign if present
                    if (i < len(word_to_process) and
                            word_to_process[i] in vowel_signs):
                        vowel = vowel_signs[word_to_process[i]]
                        if word_to_process[i] in orthographic_corrections:
                            incorrect, correct = orthographic_corrections[
                                word_to_process[i]]
                            if vowel == incorrect:
                                vowel = correct
                        translated_word += vowel
                        i += 1
                    continue

                char = word_to_process[i]

                # Map numerals directly
                if char in numerals:
                    translated_word += numerals[char]
                    i += 1
                    continue

                # Map punctuation directly
                if char in punctuation:
                    translated_word += punctuation[char]
                    i += 1
                    continue

                # Map independent vowels
                if char in vowels:
                    translated_word += vowels[char]
                    i += 1
                    continue

                # Map diacritics (anusvāra, visarga, candrabindu)
                if char in ['\U00011000', '\U00011001', '\U00011002']:
                    translated_word += diacritics[char]
                    i += 1
                    continue

                # Handle consonants and potential conjuncts
                if char in consonants:
                    curr_cons = consonants[char]
                    i += 1

                    # Check for virama indicating a conjunct
                    if (i < len(word_to_process) and
                            word_to_process[i] == '\U00011046'):
                        i += 1
                        if (i < len(word_to_process) and
                                word_to_process[i] in consonants):
                            next_cons = consonants[word_to_process[i]]
                            if curr_cons == 'c' and next_cons == 'c':
                                translated_word += 'cc'
                            else:
                                translated_word += curr_cons + next_cons
                            i += 1
                            # Add vowel sign or inherent 'a' if needed
                            if (i < len(word_to_process) and
                                    word_to_process[i] in vowel_signs):
                                vowel = vowel_signs[word_to_process[i]]
                                if word_to_process[i] in orthographic_corrections:
                                    incorrect, correct = orthographic_corrections[
                                        word_to_process[i]]
                                    if vowel == incorrect:
                                        vowel = correct
                                translated_word += vowel
                                i += 1
                            elif (i >= len(word_to_process) or
                                  word_to_process[i] != '\U00011046'):
                                translated_word += 'a'
                        else:
                            translated_word += curr_cons
                    else:
                        # Add vowel sign or inherent 'a'
                        if (i < len(word_to_process) and
                                word_to_process[i] in vowel_signs):
                            vowel = vowel_signs[word_to_process[i]]
                            if word_to_process[i] in orthographic_corrections:
                                incorrect, correct = orthographic_corrections[
                                    word_to_process[i]]
                                if vowel == incorrect:
                                    vowel = correct
                            translated_word += curr_cons + vowel
                            i += 1
                        else:
                            translated_word += curr_cons + 'a'
                    continue

                # Handle standalone vowel signs
                if char in vowel_signs:
                    vowel = vowel_signs[char]
                    if char in orthographic_corrections:
                        incorrect, correct = orthographic_corrections[char]
                        if vowel == incorrect:
                            vowel = correct
                    translated_word += vowel
                    i += 1
                    continue

                # Skip standalone virama
                if char == '\U00011046':
                    i += 1
                    continue

                # Check for unrecognized Brahmi characters
                if '\U00011000' <= char <= '\U0001107F' and char not in all_chars:
                    raise ValueError(f"Unrecognized Brahmi character: {char} "
                                     f"(Unicode: {hex(ord(char))})")

                # Pass through unrecognized non-Brahmi characters
                translated_word += char
                i += 1

            # Remove inherent 'a' if final virama is present
            if has_final_virama and translated_word.endswith('a'):
                translated_word = translated_word[:-1]

            result.append(translated_word)

        return ' '.join(result)

    except Exception as e:
        raise ValueError(f"Error transliterating Brahmi text: {str(e)}")


def analyze_complex_word(brahmi_word):
    """Analyze a Brahmi word to show its components for debugging.

    Args:
        brahmi_word (str): A single Brahmi word.

    Returns:
        dict: Analysis of components and transliteration.
    """
    components = []
    for char in brahmi_word:
        if char in ('\U00011005\U00011006\U00011007\U00011008\U00011009'
                    '\U0001100A\U0001100B\U0001100C\U0001100D\U0001100E'
                    '\U0001100F\U00011010\U00011011'):
            components.append(('vowel', char, hex(ord(char))))
        elif char in ('\U00011013\U00011014\U00011015\U00011016\U00011017'
                      '\U00011018\U00011019\U0001101A\U0001101B\U0001101C'
                      '\U0001101D\U0001101E\U0001101F\U00011020\U00011021'
                      '\U00011022\U00011023\U00011024\U00011025\U00011026'
                      '\U00011027\U00011028\U00011029\U0001102A\U0001102B'
                      '\U0001102C\U0001102D\U0001102E\U0001102F\U00011030'
                      '\U00011031\U00011032\U00011033'):
            components.append(('consonant', char, hex(ord(char))))
        elif char in ('\U00011038\U00011039\U0001103A\U0001103B\U0001103C'
                      '\U0001103D\U0001103E\U0001103F\U00011040\U00011041'
                      '\U00011042\U00011043\U00011044'):
            components.append(('vowel_sign', char, hex(ord(char))))
        elif char in '\U00011000\U00011001\U00011002\U00011046':
            components.append(('diacritic', char, hex(ord(char))))
        elif char in '\U0001104B\U0001104C\U00011047\U00011048':
            components.append(('punctuation', char, hex(ord(char))))
        elif char in ('\U00011066\U00011067\U00011068\U00011069\U0001106A'
                      '\U0001106B\U0001106C\U0001106D\U0001106E\U0001106F'):
            components.append(('numeral', char, hex(ord(char))))
        else:
            components.append(('unknown', char, hex(ord(char))))
    return {
        'word': brahmi_word,
        'components': components,
        'transliteration': brahmi_to_iast(brahmi_word)
    }


def create_test_suite():
    """Create a test suite with expected outputs."""
    return [
        ("𑀫𑀫 𑀦𑀸𑀫 𑀧𑀭𑀕 𑀅𑀲𑁆𑀢𑀺", "mama nāma paraga asti"),
        ("𑀅𑀦𑀺𑀭𑁄𑀥𑀫𑁆", "anirodham"),
        ("𑀅𑀦𑀼𑀢𑁆𑀧𑀸𑀤𑀫𑁆", "anutpādam"),
        ("𑀧𑁆𑀭𑀢𑀻𑀢𑁆𑀬𑀲𑀫𑀼𑀢𑁆𑀧𑀸𑀤𑀁", "pratītyasamutpādaṃ"),
        ("𑀅𑀦𑀼𑀘𑁆𑀙𑁂𑀤𑀫𑁆", "anucchedam"),
        ("𑀲𑀁𑀩𑀼𑀤𑁆𑀥𑀲𑁆𑀢𑀁", "saṃbuddhastaṃ"),
        ("𑀬𑀂", "yaḥ"),
        ("𑀧𑁆𑀭𑀧𑀜𑁆𑀘𑁄𑀧𑀰𑀫𑀁", "prapañcopaśamaṃ"),
        ("𑀓𑁆𑀱𑁆𑀡", "kṣṇa"),
        ("𑀲𑀁𑀂", "saṃḥ"),
        ("𑀧𑁆𑀭𑀓𑁆𑀱𑁆𑀢𑀸𑀲𑁆𑀬", "prakṣtāsya"),
        ("𑀲𑀁𑀁𑀂", "saṃṃḥ"),
        ("𑀓𑁆𑀱𑁆𑀢𑁆𑀭", "kṣtra")
    ]


if __name__ == "__main__":
    test_cases = create_test_suite()
    print("=== Brahmi to IAST Transliteration Tests ===")
    all_passed = True

    for i, (brahmi, expected) in enumerate(test_cases):
        actual = brahmi_to_iast(brahmi)
        passed = actual == expected
        all_passed = all_passed and passed

        print(f"\nTest {i+1}:")
        print(f"Original Brahmi: {brahmi}")
        print(f"Expected IAST:   {expected}")
        print(f"Actual IAST:     {actual}")
        print(f"Status: {'✅ PASS' if passed else '❌ FAIL'}")

        if not passed:
            analysis = analyze_complex_word(brahmi)
            print(f"Analysis: {analysis['components']}")

    print(f"\n=== Overall Result: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'} ===")

    numeral_test = "𑁦𑁧𑁨𑁩𑁪𑁫𑁬𑁭𑁮𑁯"
    print(f"\nNumeral Test:")
    print(f"Original Brahmi: {numeral_test}")
    print(f"IAST Transliteration: {brahmi_to_iast(numeral_test)}")

=== Brahmi to IAST Transliteration Tests ===

Test 1:
Original Brahmi: 𑀫𑀫 𑀦𑀸𑀫 𑀧𑀭𑀕 𑀅𑀲𑁆𑀢𑀺
Expected IAST:   mama nāma paraga asti
Actual IAST:     mama nāma paraga asti
Status: ✅ PASS

Test 2:
Original Brahmi: 𑀅𑀦𑀺𑀭𑁄𑀥𑀫𑁆
Expected IAST:   anirodham
Actual IAST:     anirodham
Status: ✅ PASS

Test 3:
Original Brahmi: 𑀅𑀦𑀼𑀢𑁆𑀧𑀸𑀤𑀫𑁆
Expected IAST:   anutpādam
Actual IAST:     anutpādam
Status: ✅ PASS

Test 4:
Original Brahmi: 𑀧𑁆𑀭𑀢𑀻𑀢𑁆𑀬𑀲𑀫𑀼𑀢𑁆𑀧𑀸𑀤𑀁
Expected IAST:   pratītyasamutpādaṃ
Actual IAST:     pratītyasamutpādaṃ
Status: ✅ PASS

Test 5:
Original Brahmi: 𑀅𑀦𑀼𑀘𑁆𑀙𑁂𑀤𑀫𑁆
Expected IAST:   anucchedam
Actual IAST:     anucchedam
Status: ✅ PASS

Test 6:
Original Brahmi: 𑀲𑀁𑀩𑀼𑀤𑁆𑀥𑀲𑁆𑀢𑀁
Expected IAST:   saṃbuddhastaṃ
Actual IAST:     saṃbuddhastaṃ
Status: ✅ PASS

Test 7:
Original Brahmi: 𑀬𑀂
Expected IAST:   yaḥ
Actual IAST:     yaḥ
Status: ✅ PASS

Test 8:
Original Brahmi: 𑀧𑁆𑀭𑀧𑀜𑁆𑀘𑁄𑀧𑀰𑀫𑀁
Expected IAST:   prapañcopaśamaṃ
Actual IAST:     prapañcopaśamaṃ
Status: ✅ PASS

Test 9:
Original Brahmi: 𑀓𑁆𑀱𑁆𑀡
Expected IAST:   